In [1]:
###############################################################
# IFCS 2026 DATA CHALLENGE
# Exploratory clustering of Italian SMEs
###############################################################

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score

import umap
import hdbscan

###############################################################
# Load
###############################################################

df = pd.read_csv("train.csv")

###############################################################
# Drop IDs
###############################################################

df = df.drop(columns=["Company ID"])

###############################################################
# Identify columns
###############################################################

target = "Financial distress"

categorical = [
    "Province",
    "Ateco",
    "sector"
]

numerical = [
    c for c in df.columns
    if c not in categorical + [target]
]

###############################################################
# Log-transform skewed financial variables
###############################################################

for c in numerical:
    if (df[c] > 0).mean() > 0.9:
        df[c] = np.log1p(df[c])

###############################################################
# Preprocessing
###############################################################

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numerical),
    ("cat", cat_pipe, categorical)
])

X = preprocessor.fit_transform(df)

###############################################################
# UMAP
###############################################################

embedding = umap.UMAP(
    n_neighbors=30,
    min_dist=0.05,
    random_state=42
).fit_transform(X)

###############################################################
# HDBSCAN
###############################################################

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=80,
    min_samples=20,
    prediction_data=True
)

labels = clusterer.fit_predict(embedding)

df["Cluster"] = labels

###############################################################
# Quality
###############################################################

mask = labels != -1

if len(np.unique(labels[mask])) > 1:
    sil = silhouette_score(embedding[mask], labels[mask])
    print("Silhouette =", sil)

print(df["Cluster"].value_counts())

###############################################################
# UMAP plot
###############################################################

plt.figure(figsize=(10,8))

plt.scatter(
    embedding[:,0],
    embedding[:,1],
    c=labels,
    cmap="tab20",
    s=10
)

plt.title("HDBSCAN Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")

plt.show()

###############################################################
# Cluster profiles
###############################################################

profile = (
    df
    .groupby("Cluster")[numerical]
    .median()
)

print(profile)

###############################################################
# Financial distress by cluster
###############################################################

distress = (
    df
    .groupby("Cluster")[target]
    .mean()
)

plt.figure(figsize=(8,4))
distress.sort_values().plot.bar()
plt.ylabel("Financial distress rate")
plt.show()

###############################################################
# Sector composition
###############################################################

sector_table = pd.crosstab(
    df.Cluster,
    df.sector,
    normalize="index"
)

plt.figure(figsize=(12,6))
sns.heatmap(
    sector_table,
    cmap="Blues"
)

plt.title("Sector composition")
plt.show()

###############################################################
# Province composition
###############################################################

province_table = pd.crosstab(
    df.Cluster,
    df.Province,
    normalize="index"
)

plt.figure(figsize=(15,5))

sns.heatmap(
    province_table,
    cmap="viridis"
)

plt.title("Province distribution")
plt.show()

###############################################################
# Most characteristic features
###############################################################

global_mean = df[numerical].median()

for cluster in sorted(df.Cluster.unique()):

    if cluster == -1:
        continue

    local = (
        df[df.Cluster==cluster][numerical]
        .median()
    )

    diff = (local-global_mean).abs().sort_values(ascending=False)

    print("\n")
    print("="*60)
    print("Cluster",cluster)
    print(diff.head(10))

###############################################################
# Cluster sizes
###############################################################

plt.figure(figsize=(8,4))

df.Cluster.value_counts().sort_index().plot.bar()

plt.title("Cluster sizes")

plt.show()

ImportError: Numba needs NumPy 2.2 or less. Got NumPy 2.4.